In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"


In [2]:
import datasets
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import copy

In [3]:
from transformers import AutoTokenizer

In [4]:
!nvidia-smi


Thu Feb 26 21:15:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:2D:00.0 Off |                    0 |
| N/A   40C    P0            160W /  700W |    1958MiB /  81559MiB |     29%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# EDA for GovRepQs

In [5]:
gr = datasets.load_dataset("launch/gov_report_qs", name="document", trust_remote_code=True)

In [12]:
gr_orig = datasets.load_dataset("launch/gov_report", name="plain_text", trust_remote_code=True)

README.md: 0.00B [00:00, ?B/s]

gov_report.py: 0.00B [00:00, ?B/s]

gao_train.jsonl:   0%|          | 0.00/709M [00:00<?, ?B/s]

gao_valid.jsonl:   0%|          | 0.00/41.6M [00:00<?, ?B/s]

gao_test.jsonl:   0%|          | 0.00/40.0M [00:00<?, ?B/s]

...

crs_valid.jsonl:   0%|          | 0.00/22.5M [00:00<?, ?B/s]

crs_test.jsonl:   0%|          | 0.00/19.8M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

## Merge with docs from GovReport

In [42]:
for split in gr.keys():
    val_ids = gr[split]["doc_id"]
    val_texts = gr_orig.filter(lambda x: x["id"] in val_ids)
    concated_texts = datasets.concatenate_datasets([val for key, val in val_texts.items()])
    ids_val_ids = [concated_texts["id"].index(el) for el in val_ids]
    concated_texts_sort = concated_texts.select(ids_val_ids)
    gr[split] = gr[split].add_column("document", concated_texts_sort["document"])

Filter:   0%|          | 0/17519 [00:00<?, ? examples/s]

Filter:   0%|          | 0/974 [00:00<?, ? examples/s]

Filter:   0%|          | 0/973 [00:00<?, ? examples/s]

Filter:   0%|          | 0/17519 [00:00<?, ? examples/s]

Filter:   0%|          | 0/974 [00:00<?, ? examples/s]

Filter:   0%|          | 0/973 [00:00<?, ? examples/s]

In [48]:
gr.save_to_disk("../../data/gov_report/gr_merged")

Saving the dataset (0/1 shards):   0%|          | 0/1371 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/171 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/172 [00:00<?, ? examples/s]

## Estimate lengths

In [5]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [6]:
def caclulate_length(example):
    messages = [
        {"role": "user", "content": example["document"]}
    ]
    length = len(tokenizer.apply_chat_template(messages, tokenize=True))
    example["length"] = length
    return example

In [62]:
gr = gr.map(caclulate_length)

Map:   0%|          | 0/1371 [00:00<?, ? examples/s]

Map:   0%|          | 0/171 [00:00<?, ? examples/s]

Map:   0%|          | 0/172 [00:00<?, ? examples/s]

In [63]:
train_lens = gr["train"]["length"]

In [68]:
np.percentile(train_lens, 10)

np.float64(4906.0)

In [71]:
samples_num = sum([len(el) for el in gr["train"]["question_summary_pairs"]])
samples_num

5484

In [110]:
# check if paragraphs+titles are equal to full text
diffs = []
rdiffs = []
for idx in range(len(gr["validation"])):
    sample_doc = [title.strip("\n") + par.strip("\n") for title, par in zip(gr["validation"]["document_sections"][idx]["title"], gr["validation"]["document_sections"][idx]["paragraphs"])]
    sample_doc = " ".join(sample_doc)
    diffs.append(-(len(sample_doc) - len(gr["validation"]["document"][idx])))
    rdiffs.append(diffs[-1] / len(gr["validation"]["document"][idx]))
    

In [113]:
min(diffs), max(diffs)

(5, 126)

In [114]:
gr["validation"]["question_summary_pairs"][0]

{'question': ['What does this report examine?',
  'Why might projections be inaccurate?',
  'How might these studies be helpful?',
  'By what would the ultimate cost of S. 2191 be determined?',
  "Why is it hard to project technology's potential to reduce S. 2191's costs?",
  'What do the cases suggest regarding the use of CCS bonus allowances?',
  "How is S. 2191's climate-related benefit best considered?",
  'What efforts did the US make to engage with the rest of the world on this issue?',
  'What issues does this global context raise?'],
 'summary': ['This report examines six studies that project the costs of S. 2191 (S. 3036) to 2030 or 2050.',
  'It is difficult to project costs up to the year 2030, much less beyond. The already tenuous assumption that regulatory standards will remain constant becomes more unrealistic, and other unforeseen events loom as critical issues which cannot be modeled. Long-term cost projections are at best speculative, and should be viewed with attentiv

In [116]:
gr["validation"]["document_sections"][0]

{'title': ['',
  'Overview of the Major Provisions of S. 2191 (S. 3036)',
  'Earlier Versions of the Bill',
  'Bill as Introduced',
  'Bill as Reported by Subcommittee',
  'As Ordered Reported by Committee',
  'Deficit Reduction Amendment',
  'Introduction: Models Cannot Predict the Future Costs of a Climate Change Program',
  'Lessons from SO2 Cap and Trade Program',
  'An Illustrative Example from Analyses of S. 2191',
  'Likelihood for More Noise in Greenhouse Gas Reduction Cost Estimates',
  'Complexity of the Problem',
  'Flexibility of Cap-and-Trade Program',
  'Importance of Technology to Future Results',
  'Increasing Problems with Ceteris Paribus Analysis31',
  'Changing Baselines By Changing Laws',
  'Changing Baselines By Changing Regulation35',
  'Measuring the Noise: A Web of Cost Measures',
  'Three Perspectives: Getting Out of the Noise',
  'Results for S. 2191',
  'Impact on Greenhouse Gas Emissions',
  'Impact on Non-Greenhouse Gas Emissions',
  'Impact on GDP Per Capi

In [109]:
len(sample_doc), len(gr["validation"]["document"][idx])

(34751, 34762)

In [97]:
sample_doc

'As Congress continues the debate on an appropriate response to the climate change issue, multiple bills have been introduced to begin reducing greenhouse gas (GHG) emissions. Of these, S. 2191 (the Lieberman-Warner Climate Security Act of 2008 ) has received particular attention. Introduced by Senator Lieberman, S. 2191 was ordered reported by the Senate Committee on Environment and Public Works on December 5, 2007. Numerous analyses have been done on its impacts, and as of April 2008, six studies had been released.\nThe most comprehensive analysis has been conducted by the U.S. Environmental Protection Agency (EPA). The report is entitled: EPA Analysis of the Lieberman-Warner Climate Security Act of 2008: S. 2191 in 110 th Congress (March 14, 2008). The analysis employs a suite of models and basecases, along with some useful sensitivity analyses. This report will focus on three of the models, two basecases, and sensitivity analysis as appropriate.\nThe first model is ADAGE: a computa

In [100]:
gr["validation"]["document"][idx]

'As Congress continues the debate on an appropriate response to the climate change issue, multiple bills have been introduced to begin reducing greenhouse gas (GHG) emissions. Of these, S. 2191 (the Lieberman-Warner Climate Security Act of 2008 ) has received particular attention. Introduced by Senator Lieberman, S. 2191 was ordered reported by the Senate Committee on Environment and Public Works on December 5, 2007. Numerous analyses have been done on its impacts, and as of April 2008, six studies had been released. The most comprehensive analysis has been conducted by the U.S. Environmental Protection Agency (EPA). The report is entitled: EPA Analysis of the Lieberman-Warner Climate Security Act of 2008: S. 2191 in 110 th Congress (March 14, 2008). The analysis employs a suite of models and basecases, along with some useful sensitivity analyses. This report will focus on three of the models, two basecases, and sensitivity analysis as appropriate. The first model is ADAGE: a computabl

In [101]:
len(sample_doc), len(gr["validation"]["document"][idx])

(123980, 124023)

In [50]:
gr["validation"]["question_summary_pairs"][0]

{'question': ['What does this report examine?',
  'Why might projections be inaccurate?',
  'How might these studies be helpful?',
  'By what would the ultimate cost of S. 2191 be determined?',
  "Why is it hard to project technology's potential to reduce S. 2191's costs?",
  'What do the cases suggest regarding the use of CCS bonus allowances?',
  "How is S. 2191's climate-related benefit best considered?",
  'What efforts did the US make to engage with the rest of the world on this issue?',
  'What issues does this global context raise?'],
 'summary': ['This report examines six studies that project the costs of S. 2191 (S. 3036) to 2030 or 2050.',
  'It is difficult to project costs up to the year 2030, much less beyond. The already tenuous assumption that regulatory standards will remain constant becomes more unrealistic, and other unforeseen events loom as critical issues which cannot be modeled. Long-term cost projections are at best speculative, and should be viewed with attentiv

In [47]:
gr["validation"]

Dataset({
    features: ['doc_id', 'document_sections', 'question_summary_pairs', 'document'],
    num_rows: 171
})

In [13]:
val_ids = gr["validation"]["doc_id"]

In [18]:
val_texts = gr_orig['validation'].filter(lambda x: x["id"] in val_ids)#["document"]

In [26]:
val_texts = gr_orig.filter(lambda x: x["id"] in val_ids)#["document"]

In [32]:
concated_texts = datasets.concatenate_datasets([val for key, val in val_texts.items()])

In [38]:
ids_val_ids = [concated_texts["id"].index(el) for el in val_ids]

In [40]:
concated_texts_sort = concated_texts.select(ids_val_ids)

In [33]:
concated_texts

Dataset({
    features: ['id', 'document', 'summary'],
    num_rows: 171
})

In [41]:
concated_texts_sort["id"]

['CRS_RL34489',
 'CRS_R42054',
 'GAO_GAO-16-396',
 'CRS_R42766',
 'CRS_RL33087',
 'GAO_GAO-16-46',
 'GAO_GAO-13-77',
 'GAO_GAO-13-260',
 'GAO_GAO-12-58',
 'GAO_GAO-12-712',
 'GAO_GAO-14-54',
 'GAO_GAO-19-520',
 'CRS_R42717',
 'GAO_GAO-14-364T',
 'CRS_RS22549',
 'GAO_GAO-13-620',
 'GAO_GAO-14-369',
 'CRS_R44890',
 'GAO_GAO-17-607',
 'GAO_GAO-15-271',
 'CRS_R45422',
 'GAO_GAO-17-483',
 'GAO_GAO-15-217',
 'CRS_R42590',
 'GAO_GAO-13-682',
 'CRS_R45261',
 'CRS_RL34577',
 'CRS_RL32303',
 'GAO_GAO-14-67',
 'CRS_R44481',
 'GAO_GAO-18-54',
 'CRS_R42789',
 'GAO_GAO-13-789',
 'GAO_GAO-16-166',
 'GAO_GAO-16-235',
 'GAO_GAO-14-602',
 'GAO_GAO-12-875',
 'CRS_R41985',
 'CRS_R40656',
 'GAO_GAO-12-319',
 'CRS_R43620',
 'CRS_R42561',
 'CRS_R43555',
 'CRS_RL33311',
 'CRS_RL31932',
 'GAO_GAO-18-224',
 'CRS_R43218',
 'GAO_GAO-13-347T',
 'GAO_GAO-18-473',
 'CRS_R44715',
 'GAO_GAO-16-438',
 'CRS_RL32952',
 'GAO_GAO-16-453',
 'GAO_GAO-17-352',
 'CRS_R41588',
 'GAO_GAO-14-356',
 'CRS_R40193',
 'CRS_R41991',
 '

In [34]:
val_ids

['CRS_RL34489',
 'CRS_R42054',
 'GAO_GAO-16-396',
 'CRS_R42766',
 'CRS_RL33087',
 'GAO_GAO-16-46',
 'GAO_GAO-13-77',
 'GAO_GAO-13-260',
 'GAO_GAO-12-58',
 'GAO_GAO-12-712',
 'GAO_GAO-14-54',
 'GAO_GAO-19-520',
 'CRS_R42717',
 'GAO_GAO-14-364T',
 'CRS_RS22549',
 'GAO_GAO-13-620',
 'GAO_GAO-14-369',
 'CRS_R44890',
 'GAO_GAO-17-607',
 'GAO_GAO-15-271',
 'CRS_R45422',
 'GAO_GAO-17-483',
 'GAO_GAO-15-217',
 'CRS_R42590',
 'GAO_GAO-13-682',
 'CRS_R45261',
 'CRS_RL34577',
 'CRS_RL32303',
 'GAO_GAO-14-67',
 'CRS_R44481',
 'GAO_GAO-18-54',
 'CRS_R42789',
 'GAO_GAO-13-789',
 'GAO_GAO-16-166',
 'GAO_GAO-16-235',
 'GAO_GAO-14-602',
 'GAO_GAO-12-875',
 'CRS_R41985',
 'CRS_R40656',
 'GAO_GAO-12-319',
 'CRS_R43620',
 'CRS_R42561',
 'CRS_R43555',
 'CRS_RL33311',
 'CRS_RL31932',
 'GAO_GAO-18-224',
 'CRS_R43218',
 'GAO_GAO-13-347T',
 'GAO_GAO-18-473',
 'CRS_R44715',
 'GAO_GAO-16-438',
 'CRS_RL32952',
 'GAO_GAO-16-453',
 'GAO_GAO-17-352',
 'CRS_R41588',
 'GAO_GAO-14-356',
 'CRS_R40193',
 'CRS_R41991',
 '

In [25]:
gr["validation"]#["doc_id"]

Dataset({
    features: ['doc_id', 'document_sections', 'question_summary_pairs'],
    num_rows: 171
})

In [6]:
x=1

In [7]:
gr["validation"]

Dataset({
    features: ['doc_id', 'document_sections', 'question_summary_pairs'],
    num_rows: 171
})

In [11]:
len(gr["validation"]["document_sections"][0]["title"]), len(gr["validation"]["document_sections"][0]["paragraphs"]),

(50, 50)

In [10]:
gr["validation"]["document_sections"][0]

{'title': ['',
  'Overview of the Major Provisions of S. 2191 (S. 3036)',
  'Earlier Versions of the Bill',
  'Bill as Introduced',
  'Bill as Reported by Subcommittee',
  'As Ordered Reported by Committee',
  'Deficit Reduction Amendment',
  'Introduction: Models Cannot Predict the Future Costs of a Climate Change Program',
  'Lessons from SO2 Cap and Trade Program',
  'An Illustrative Example from Analyses of S. 2191',
  'Likelihood for More Noise in Greenhouse Gas Reduction Cost Estimates',
  'Complexity of the Problem',
  'Flexibility of Cap-and-Trade Program',
  'Importance of Technology to Future Results',
  'Increasing Problems with Ceteris Paribus Analysis31',
  'Changing Baselines By Changing Laws',
  'Changing Baselines By Changing Regulation35',
  'Measuring the Noise: A Web of Cost Measures',
  'Three Perspectives: Getting Out of the Noise',
  'Results for S. 2191',
  'Impact on Greenhouse Gas Emissions',
  'Impact on Non-Greenhouse Gas Emissions',
  'Impact on GDP Per Capi

# Build GovRep by splits

In [7]:
dat = datasets.load_from_disk("../../data/gov_report/gr_merged")

In [8]:
dat

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'document_sections', 'question_summary_pairs', 'document'],
        num_rows: 1371
    })
    validation: Dataset({
        features: ['doc_id', 'document_sections', 'question_summary_pairs', 'document'],
        num_rows: 171
    })
    test: Dataset({
        features: ['doc_id', 'document_sections', 'question_summary_pairs', 'document'],
        num_rows: 172
    })
})

## Match gold paragraphs

In [9]:
from functools import partial
from tqdm import tqdm
import copy

In [10]:
par_subset = datasets.load_dataset("launch/gov_report_qs", name="paragraph", trust_remote_code=True)

In [11]:
def add_gold_pars(example, split):
    par_samples = par_subset[split].filter(lambda x: x["doc_id"] == example["doc_id"])
    #titles = [par_sample["title"] for par_sample in par_samples["document_sections"]]
    #pars = [par_sample["paragraphs"] for par_sample in par_samples["document_sections"]]
    #quests = [par_sample["question"] for par_sample in par_samples["question_summary_pairs"]]
    #summs = [par_sample["summary"] for par_sample in par_samples["question_summary_pairs"]]
    example["golden_pars"] = {
        "titles": [par_sample["title"] for par_sample in par_samples["document_sections"]],
        "paragraphs": [par_sample["paragraphs"] for par_sample in par_samples["document_sections"]],
        "questions": [par_sample["question"] for par_sample in par_samples["question_summary_pairs"]],
        "summaries": [par_sample["summary"] for par_sample in par_samples["question_summary_pairs"]],
    }
    return example

In [10]:
dat_with_gold = copy.deepcopy(dat)
for split in dat.keys():
    dat_with_gold[split] = dat[split].map(partial(add_gold_pars, split=split))

Map:   0%|          | 0/1371 [00:00<?, ? examples/s]

Map:   0%|          | 0/171 [00:00<?, ? examples/s]

Map:   0%|          | 0/172 [00:00<?, ? examples/s]

In [31]:
dat_with_gold.save_to_disk("../../data/gov_report/gr_merged_with_gold")

Saving the dataset (0/1 shards):   0%|          | 0/1371 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/171 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/172 [00:00<?, ? examples/s]

In [11]:
dat_with_gold

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'document_sections', 'question_summary_pairs', 'document', 'golden_pars'],
        num_rows: 1371
    })
    validation: Dataset({
        features: ['doc_id', 'document_sections', 'question_summary_pairs', 'document', 'golden_pars'],
        num_rows: 171
    })
    test: Dataset({
        features: ['doc_id', 'document_sections', 'question_summary_pairs', 'document', 'golden_pars'],
        num_rows: 172
    })
})

In [12]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [13]:
def caclulate_par_length(example):
    par_lens = []
    for title, text in zip(example["document_sections"]["title"], example["document_sections"]["paragraphs"]):
        messages = [
            {"role": "system", "content": ''},
            {"role": "user", "content": title + '\n'+ text}
        ]
        length = len(tokenizer.apply_chat_template(messages, tokenize=True))
        par_lens.append(length)
    example["document_sections"]["par_lens"] = par_lens
    return example

In [14]:
dat_with_gold = dat_with_gold.map(caclulate_par_length)

Map:   0%|          | 0/1371 [00:00<?, ? examples/s]

Map:   0%|          | 0/171 [00:00<?, ? examples/s]

Map:   0%|          | 0/172 [00:00<?, ? examples/s]

In [14]:
def match_gold_par(example):
    gold_pars_ids = []
    for gold_pars in example["golden_pars"]["paragraphs"]:
        buf_gold_ids = []
        for gold_par in gold_pars:
            for par_idx, par in enumerate(example["document_sections"]["paragraphs"]):
                if par == gold_par:
                    buf_gold_ids.append(par_idx)
                    break
        gold_pars_ids.append(buf_gold_ids)
    example["golden_pars"]["paragraphs_ids"] = gold_pars_ids
    return example

In [16]:
dat_with_gold_ids = dat_with_gold.map(match_gold_par)

Map:   0%|          | 0/1371 [00:00<?, ? examples/s]

Map:   0%|          | 0/171 [00:00<?, ? examples/s]

Map:   0%|          | 0/172 [00:00<?, ? examples/s]

In [15]:
def sample_pars(par_ids, gold_par_ids, par_lens, max_len):
    max_iters = 20
    curr_len = sum(np.array(par_lens)[gold_par_ids])
    sample_length = np.random.randint(min(max_len-1, curr_len), max_len)
    selected_ids = copy.deepcopy(gold_par_ids)
    ids_for_sampling = [x for x in par_ids if x not in gold_par_ids]
    steps = 0
    while curr_len < sample_length and steps < max_iters and len(ids_for_sampling) > 0:
        buf_idx = int(np.random.choice(ids_for_sampling, 1, replace=False)[0])
        buf_len = curr_len + par_lens[buf_idx]
        if buf_len > sample_length:
            steps += 1
        else:
            steps = 0
            curr_len = buf_len
            selected_ids.append(buf_idx)
            ids_for_sampling.remove(buf_idx)
    return selected_ids

In [16]:
def caclulate_length_full(example):
    messages = [
        {"role": "user", "content": f"Question:\n{example['question']}\nContext:\n{example['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{example['answer']}\n"},
    ]
    length = len(tokenizer.apply_chat_template(messages, tokenize=True))
    example["length"] = length
    return example

## Build splits - from 2k to 16k

In [19]:
dataset_2k = {}
max_len = 256
save_name = "gov_report_0k_256"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 520.86it/s]


Map:   0%|          | 0/19878 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/19878 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/39 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/6 [00:00<?, ? examples/s]

In [20]:
gov_rep_2k

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 39
    })
    validation: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 7
    })
    test: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 6
    })
})

In [21]:
dataset_2k = {}
max_len = 512
save_name = "gov_report_0k_512"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 822.48it/s]


Map:   0%|          | 0/19878 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/19878 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/452 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/35 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/42 [00:00<?, ? examples/s]

In [22]:
gov_rep_2k

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 452
    })
    validation: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 35
    })
    test: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 42
    })
})

In [23]:
dataset_2k = {}
max_len = 1024
save_name = "gov_report_0k_1k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 707.83it/s]


Map:   0%|          | 0/19878 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/19878 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2915 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/146 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/351 [00:00<?, ? examples/s]

In [24]:
gov_rep_2k

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 2915
    })
    validation: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 146
    })
    test: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 351
    })
})

In [115]:
dataset_2k = {}
max_len = 2048
save_name = "gov_report_0k_2k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 398.19it/s]


Map:   0%|          | 0/19878 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/19878 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9903 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/483 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1259 [00:00<?, ? examples/s]

In [116]:
dataset_2k = {}
max_len = 4096
save_name = "gov_report_0k_4k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report/{save_name}")


100%|██████████| 172/172 [00:01<00:00, 147.72it/s]


Map:   0%|          | 0/19878 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/19878 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/17450 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/876 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2215 [00:00<?, ? examples/s]

In [117]:
dataset_2k = {}
max_len = 8192
save_name = "gov_report_0k_8k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 250.66it/s]


Map:   0%|          | 0/19878 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/19878 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/19682 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/988 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2543 [00:00<?, ? examples/s]

In [118]:
dataset_2k = {}
max_len = 4096*4
save_name = "gov_report_0k_16k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report/{save_name}")


100%|██████████| 172/172 [00:01<00:00, 94.86it/s] 


Map:   0%|          | 0/19878 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/19878 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/19843 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2592 [00:00<?, ? examples/s]

In [119]:
dataset_2k = {}
max_len = 1024*32
save_name = "gov_report_0k_32k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 187.60it/s]


Map:   0%|          | 0/19878 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/19878 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/19872 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2592 [00:00<?, ? examples/s]

In [120]:
dataset_2k = {}
max_len = 1024*64
save_name = "gov_report_0k_64k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 182.79it/s]


Map:   0%|          | 0/19878 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/19878 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/19878 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2592 [00:00<?, ? examples/s]

In [121]:
dataset_2k = {}
max_len = 1024*128
save_name = "gov_report_0k_128k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 177.12it/s]


Map:   0%|          | 0/19878 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/19878 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/19878 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2592 [00:00<?, ? examples/s]

## Build test sets

In [125]:
np.random.seed(43)
max_lens = [2, 4, 8, 16, 32, 64]#, 128]
min_lens = [0, 2, 4, 8, 16, 32]#, 64]
for min_len_k, max_len_k in zip(min_lens, max_lens):
    gov_rep_2k_test = datasets.load_from_disk(f"../../data/gov_report/gov_report_0k_{max_len_k}k")
    test_set = gov_rep_2k_test["test"]
    # filter samples by length
    test_set = test_set.filter(lambda x: x["length"] < max_len_k*1024 and x["length"] > min_len_k*1024)
    # select 100 samples
    test_ids = np.random.choice(np.arange(len(test_set)), min(100, len(test_set)), replace=False)
    test_subset = test_set.select(test_ids)
    test_subset.save_to_disk(f"../../data/gov_report/gov_report_{min_len_k}k_{max_len_k}k_test_only")
    print(min_len_k, max_len_k, len(test_set), len(test_subset))

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

0 2 1259 100


Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

2 4 1759 100


Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

...

8 16 983 100


Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

16 32 304 100


Saving the dataset (0/1 shards):   0%|          | 0/26 [00:00<?, ? examples/s]

32 64 26 26


In [127]:
# join test datasets
full_test = []
for min_len_k, max_len_k in zip(min_lens, max_lens):
    gov_rep_2k_test = datasets.load_from_disk(f"../../data/gov_report/gov_report_{min_len_k}k_{max_len_k}k_test_only")
    test_set = gov_rep_2k_test
    full_test.append(test_set)
full_test = datasets.concatenate_datasets(full_test)

In [147]:
full_test.save_to_disk(f"../../data/gov_report/gov_report_0k_64k_test_only")

Saving the dataset (0/1 shards):   0%|          | 0/526 [00:00<?, ? examples/s]

## Build test sets - rebalanced support

In [8]:
np.random.seed(43)
max_lens = [1, 2, 4, 6, 8, 10, 12, 14, 16, 24, 32, 48, 64]#, 128]
min_lens = [0, 1, 2, 4, 6, 8, 10, 12, 14, 16, 24, 32, 48]#, 64]
max_len_ds = [2, 2, 4, 8, 8, 16, 16, 16, 16, 32, 32, 64, 64]#, 128]
for idx, (min_len_k, max_len_k) in enumerate(zip(min_lens, max_lens)):
    max_len_dataset = max_len_ds[idx]
    gov_rep_2k_test = datasets.load_from_disk(f"../../data/gov_report/gov_report_0k_{max_len_dataset}k")
    test_set = gov_rep_2k_test["test"]
    # filter samples by length
    test_set = test_set.filter(lambda x: x["length"] < max_len_k*1024 and x["length"] > min_len_k*1024)
    # select 100 samples
    test_ids = np.random.choice(np.arange(len(test_set)), min(100, len(test_set)), replace=False)
    test_subset = test_set.select(test_ids)
    test_subset.save_to_disk(f"../../data/gov_report/gov_report_{min_len_k}k_{max_len_k}k_test_only_bal_support")
    print(min_len_k, max_len_k, len(test_set), len(test_subset))

Filter:   0%|          | 0/1259 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

0 1 117 100


Filter:   0%|          | 0/1259 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

...

Saving the dataset (0/1 shards):   0%|          | 0/20 [00:00<?, ? examples/s]

32 48 20 20


Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/6 [00:00<?, ? examples/s]

48 64 6 6


In [9]:
# join test datasets
full_test = []
for min_len_k, max_len_k in zip(min_lens, max_lens):
    gov_rep_2k_test = datasets.load_from_disk(f"../../data/gov_report/gov_report_{min_len_k}k_{max_len_k}k_test_only_bal_support")
    test_set = gov_rep_2k_test
    full_test.append(test_set)
full_test = datasets.concatenate_datasets(full_test)

In [10]:
full_test.save_to_disk(f"../../data/gov_report/gov_report_0k_64k_test_only_bal_support")

Saving the dataset (0/1 shards):   0%|          | 0/1049 [00:00<?, ? examples/s]

In [11]:
full_test

Dataset({
    features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
    num_rows: 1049
})

In [132]:
answ_toks_lens = [len(tokenizer(el)["input_ids"]) for el in full_test["answer"]]

In [146]:
np.min(answ_toks_lens), np.max(answ_toks_lens), np.mean(answ_toks_lens), np.percentile(answ_toks_lens, 99.9)

(np.int64(8),
 np.int64(185),
 np.float64(42.29277566539924),
 np.float64(165.57500000000084))

In [148]:
gov_rep_2k = datasets.load_from_disk(f"../../data/gov_report/gov_report_0k_2k")
gov_rep_2k

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 9903
    })
    validation: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 483
    })
    test: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 1259
    })
})

In [152]:
gov_rep_2k["train"][3]

{'doc_id': 'GAO_GAO-13-174',
 'context': 'Agencies Did Not Provide Milestones with Scheduled Completion Dates for Many Priority Goals\nThe act requires each APG to have clearly defined quarterly milestones— scheduled events signifying the completion of a major deliverable or a set of related deliverables or a phase of work. Similar to performance measures, OMB’s guidance states that milestones will follow fiscal year quarters and notes that agencies may choose monthly milestones if preferred. In addition, a draft guide developed by the PIC describes characteristics of a good milestone, such as articulating concrete actions to be taken and being time-bound.\nMilestones can help agencies demonstrate that they have clear and fully developed strategies and are tracking progress to accomplish their goals. Such strategies, as identified in our past work, should (1) identify specific actions agencies are taking or plan to take to carry out their missions, (2) outline planned accomplishments, 

## Upload datasets

In [156]:
# upload dataset
gov_rep_2k = datasets.load_from_disk(f"../../data/gov_report/gov_report_0k_16k")
gov_rep_2k.push_to_hub("AIRI-NLP/gov_report_0k_16k", private=True, token="")

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/AIRI-NLP/gov_report_0k_16k/commit/818a938b4e5c50cc64aede41c4ecf62966d566ce', commit_message='Upload dataset', commit_description='', oid='818a938b4e5c50cc64aede41c4ecf62966d566ce', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/AIRI-NLP/gov_report_0k_16k', endpoint='https://huggingface.co', repo_type='dataset', repo_id='AIRI-NLP/gov_report_0k_16k'), pr_revision=None, pr_num=None)

In [157]:
gov_rep_2k = datasets.load_from_disk(f"../../data/gov_report/gov_report_0k_64k_test_only")
gov_rep_2k.push_to_hub("AIRI-NLP/gov_report_0k_64k_test_only", private=True, token="")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/AIRI-NLP/gov_report_0k_64k_test_only/commit/b5a647897c7f97d624f7fbd1823e4815dfadb065', commit_message='Upload dataset', commit_description='', oid='b5a647897c7f97d624f7fbd1823e4815dfadb065', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/AIRI-NLP/gov_report_0k_64k_test_only', endpoint='https://huggingface.co', repo_type='dataset', repo_id='AIRI-NLP/gov_report_0k_64k_test_only'), pr_revision=None, pr_num=None)

In [161]:
# now load mt datasets
gov_rep_2k = datasets.load_from_disk(f"../../data/mt_prepared/manytypes_flat_stacked_no_empty_sv_0k_16k")
gov_rep_2k.push_to_hub("AIRI-NLP/manytypes_flat_stacked_no_empty_sv_0k_16k", private=True, token="")

Uploading the dataset shards:   0%|          | 0/9 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/19 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/19 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/19 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/19 [00:00<?, ?ba/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/18 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/AIRI-NLP/manytypes_flat_stacked_no_empty_sv_0k_16k/commit/f8a4e1da837f7ec08289904ecd72ef96841b61e4', commit_message='Upload dataset', commit_description='', oid='f8a4e1da837f7ec08289904ecd72ef96841b61e4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/AIRI-NLP/manytypes_flat_stacked_no_empty_sv_0k_16k', endpoint='https://huggingface.co', repo_type='dataset', repo_id='AIRI-NLP/manytypes_flat_stacked_no_empty_sv_0k_16k'), pr_revision=None, pr_num=None)

In [164]:
# now load mt datasets
gov_rep_2k = datasets.load_from_disk(f"../../data/mt_prepared/manytypes_flat_stacked_no_empty_sv_0k_8k")
gov_rep_2k.save_to_disk(f"../../data/mt_prepared/manytypes_quest_end_flat_stacked_no_empty_sv_0k_8k")

Saving the dataset (0/5 shards):   0%|          | 0/159267 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2268 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/16039 [00:00<?, ? examples/s]

## Prepare for generation

In [165]:
data_for_gen = datasets.load_from_disk("../../data/gov_report/gr_merged_with_gold")

In [168]:
data_for_gen.save_to_disk("../../data/gov_report/gr_merged_with_gold_debug")

Saving the dataset (0/1 shards):   0%|          | 0/1371 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/171 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/172 [00:00<?, ? examples/s]

In [175]:
dat_with_gold_ids.save_to_disk("../../data/gov_report/gr_merged_with_gold_with_ids_debug")

Saving the dataset (0/1 shards):   0%|          | 0/1371 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/171 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/172 [00:00<?, ? examples/s]

In [169]:
dat_with_gold_ids["validation"][0]

{'doc_id': 'CRS_RL34489',
 'document_sections': {'alignment': ['h0_title h2_title h1_title',
   '',
   '',
   '',
   '',
   '',
   '',
   'h0_title',
   'h0_full',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   'h2_title h1_title',
   'h1_title',
   'h1_title',
   'h1_full',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   'h2_title',
   'h2_title',
   '',
   'h2_full',
   '',
   '',
   'h0_full h2_full h1_full'],
  'depth': [0,
   1,
   2,
   3,
   3,
   3,
   3,
   1,
   2,
   2,
   1,
   2,
   2,
   2,
   2,
   3,
   3,
   1,
   2,
   1,
   2,
   2,
   2,
   2,
   2,
   1,
   2,
   3,
   4,
   4,
   4,
   3,
   4,
   3,
   4,
   4,
   4,
   2,
   3,
   3,
   3,
   3,
   3,
   2,
   3,
   4,
   4,
   3,
   3,
   1],
  'par_lens': [1250,
   1173,
   36,
   216,
   283,
   220,
   97,
   44,
   776,
   1369,
   142,
   591,
   338,
   257,
   233,
   534,
   445,
   1

In [170]:
dat_with_gold_ids["validation"][0].keys()

dict_keys(['doc_id', 'document_sections', 'question_summary_pairs', 'document', 'golden_pars'])

In [174]:
dat_with_gold_ids["validation"][0]["golden_pars"].keys()

dict_keys(['paragraphs', 'paragraphs_ids', 'questions', 'summaries', 'titles'])

In [179]:
dat_with_gold_ids["validation"][0]["golden_pars"]

{'paragraphs': [['During the Clean Air Act debate in 1990 on the Title IV sulfur dioxide (SO 2 ) cap-and trade program, CRS found it difficult to analyze the cost of the bill beyond the first 10 years (1990-2000), and considered any breakdown of even 2000 data on a state-by-state basis as "not useful for any more than illustrative purposes." As stated in 1990:\nIt is difficult (and some would consider it unwise) to project costs up to the year 2000, much less beyond. The already tenuous assumption that current regulatory standards will remain constant becomes more unrealistic, and other unforeseen events (such as electric utility deregulation) loom as critical issues which can not be modeled. Hence, cost projections beyond the year 2000 are at best speculative, and are more a function of each model \' s assumptions and structure than they are of the details of proposed legislation . Projections this far into the future are based more on philosophy than analysis. [emphasis in original]\

In [176]:
d11 = datasets.load_from_disk("../../data/finetune_generated/DEBUG_gov_rep/")

In [178]:
d11[0]["golden_pars"]

{'paragraphs': [['PPACA directs PCORI to enter into contracts to carry out its research priorities. PCORI has established a multi-step research funding process, which includes merit review, that is designed to select high quality research that has the best potential to improve patient outcomes, according to PCORI officials. PCORI officials stated that their merit review process is modeled on the peer review processes established by AHRQ and NIH, which are described in law. Unlike AHRQ and NIH, however, PCORI’s merit review process utilizes patients and stakeholders in the review and scoring of applications. Key steps in PCORI’s research funding process include the development and posting of funding announcements, review and scoring of applications, and final approval by the Board of Governors. (See figure 1.)\nSubmitted applications are assessed by reviewers recruited by PCORI and selected based on expertise or knowledge in a particular subject area. Reviewers may be patients, other st

## Merge with generated datasets

In [9]:
data_init = datasets.load_from_disk("../../data/gov_report/gr_merged_with_gold_with_ids_debug")

In [15]:
gen_data_pathes = [
    "../../data/finetune_generated/gov_rep_base_gen_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_7b_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_32b_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_72b_1",
    "../../data/finetune_generated/gov_rep_base_gen_gemma_2_9b_1",
    "../../data/finetune_generated/gov_rep_base_gen_phi_3_small_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_3_4b_1",
]

In [16]:
# base merging - get 50% of each model, after merge with train, build splits and save

In [34]:
all_gen_data = []
for path in gen_data_pathes:
    gen_data = datasets.load_from_disk(path)
    np.random.seed(43)
    gen_data = gen_data.select(np.random.choice(np.arange(len(gen_data)), int(0.5*len(gen_data)), replace=False))
    all_gen_data.append(gen_data)
all_gen_data = datasets.concatenate_datasets(all_gen_data)

In [35]:
for col in data_init["train"].column_names:
    all_gen_data = all_gen_data.cast_column(col, data_init["train"].features[col])

Casting the dataset:   0%|          | 0/34493 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/34493 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/34493 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/34493 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/34493 [00:00<?, ? examples/s]

In [36]:
data_init["train"].features

{'doc_id': Value(dtype='string', id=None),
 'document_sections': {'alignment': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
  'depth': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None),
  'par_lens': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None),
  'paragraphs': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
  'title': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None)},
 'question_summary_pairs': Sequence(feature={'question': Value(dtype='string', id=None), 'summary': Value(dtype='string', id=None), 'parent_pair_index': Value(dtype='int32', id=None), 'summary_paragraph_index': Value(dtype='int32', id=None)}, length=-1, id=None),
 'document': Value(dtype='string', id=None),
 'golden_pars': {'paragraphs': Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None),
  'paragraphs_ids': Sequence(feature=Sequence(feature=Value(dtype='int64

In [37]:
all_gen_data.features

{'doc_id': Value(dtype='string', id=None),
 'document_sections': {'alignment': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
  'depth': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None),
  'par_lens': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None),
  'paragraphs': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
  'title': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None)},
 'question_summary_pairs': Sequence(feature={'question': Value(dtype='string', id=None), 'summary': Value(dtype='string', id=None), 'parent_pair_index': Value(dtype='int32', id=None), 'summary_paragraph_index': Value(dtype='int32', id=None)}, length=-1, id=None),
 'document': Value(dtype='string', id=None),
 'golden_pars': {'paragraphs': Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None),
  'paragraphs_ids': Sequence(feature=Sequence(feature=Value(dtype='int64

In [38]:
data_with_gen = copy.deepcopy(data_init)
data_with_gen["train"] = datasets.concatenate_datasets([data_with_gen["train"], all_gen_data])

In [40]:
data_with_gen["train"] = data_with_gen["train"].shuffle(seed=43)

In [41]:
data_with_gen.save_to_disk("../../data/gov_report/gr_merged_with_gold_with_ids_with_50_gen")

Saving the dataset (0/10 shards):   0%|          | 0/35864 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/171 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/172 [00:00<?, ? examples/s]

## Build splits - from 2k to 16k - for gen data

In [42]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [43]:
def caclulate_par_length(example):
    par_lens = []
    for title, text in zip(example["document_sections"]["title"], example["document_sections"]["paragraphs"]):
        messages = [
            {"role": "system", "content": ''},
            {"role": "user", "content": title + '\n'+ text}
        ]
        length = len(tokenizer.apply_chat_template(messages, tokenize=True))
        par_lens.append(length)
    example["document_sections"]["par_lens"] = par_lens
    return example

In [44]:
dat_with_gold = data_with_gen

In [45]:
dat_with_gold = dat_with_gold.map(caclulate_par_length)

Map:   0%|          | 0/35864 [00:00<?, ? examples/s]

Map:   0%|          | 0/171 [00:00<?, ? examples/s]

Map:   0%|          | 0/172 [00:00<?, ? examples/s]

In [46]:
def match_gold_par(example):
    gold_pars_ids = []
    for gold_pars in example["golden_pars"]["paragraphs"]:
        buf_gold_ids = []
        for gold_par in gold_pars:
            for par_idx, par in enumerate(example["document_sections"]["paragraphs"]):
                if par == gold_par:
                    buf_gold_ids.append(par_idx)
                    break
        gold_pars_ids.append(buf_gold_ids)
    example["golden_pars"]["paragraphs_ids"] = gold_pars_ids
    return example

In [47]:
dat_with_gold_ids = dat_with_gold.map(match_gold_par)

Map:   0%|          | 0/35864 [00:00<?, ? examples/s]

Map:   0%|          | 0/171 [00:00<?, ? examples/s]

Map:   0%|          | 0/172 [00:00<?, ? examples/s]

In [48]:
def sample_pars(par_ids, gold_par_ids, par_lens, max_len):
    max_iters = 20
    curr_len = sum(np.array(par_lens)[gold_par_ids])
    sample_length = np.random.randint(min(max_len-1, curr_len), max_len)
    selected_ids = copy.deepcopy(gold_par_ids)
    ids_for_sampling = [x for x in par_ids if x not in gold_par_ids]
    steps = 0
    while curr_len < sample_length and steps < max_iters and len(ids_for_sampling) > 0:
        buf_idx = int(np.random.choice(ids_for_sampling, 1, replace=False)[0])
        buf_len = curr_len + par_lens[buf_idx]
        if buf_len > sample_length:
            steps += 1
        else:
            steps = 0
            curr_len = buf_len
            selected_ids.append(buf_idx)
            ids_for_sampling.remove(buf_idx)
    return selected_ids

In [49]:
def caclulate_length_full(example):
    messages = [
        {"role": "user", "content": f"Question:\n{example['question']}\nContext:\n{example['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{example['answer']}\n"},
    ]
    length = len(tokenizer.apply_chat_template(messages, tokenize=True))
    example["length"] = length
    return example

In [50]:
dataset_2k = {}
max_len = 2048
save_name = "gov_report_0k_2k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_50_gen/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 370.00it/s]


Map:   0%|          | 0/54371 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/54371 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/35353 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/483 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1259 [00:00<?, ? examples/s]

In [51]:
dataset_2k = {}
max_len = 4096
save_name = "gov_report_0k_4k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_50_gen/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 298.79it/s]


Map:   0%|          | 0/54371 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/54371 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/50473 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/876 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2215 [00:00<?, ? examples/s]

In [52]:
dataset_2k = {}
max_len = 8192
save_name = "gov_report_0k_8k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_50_gen/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 249.15it/s]


Map:   0%|          | 0/54371 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/54371 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/54135 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/988 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2543 [00:00<?, ? examples/s]

In [53]:
dataset_2k = {}
max_len = 4096*4
save_name = "gov_report_0k_16k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_50_gen/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 224.51it/s]


Map:   0%|          | 0/54371 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/54371 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/5 shards):   0%|          | 0/54333 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2592 [00:00<?, ? examples/s]

In [54]:
dataset_2k = {}
max_len = 1024*32
save_name = "gov_report_0k_32k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_50_gen/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 199.47it/s]


Map:   0%|          | 0/54371 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/54371 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/6 shards):   0%|          | 0/54364 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2592 [00:00<?, ? examples/s]

In [55]:
dataset_2k = {}
max_len = 1024*64
save_name = "gov_report_0k_64k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_50_gen/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 189.91it/s]


Map:   0%|          | 0/54371 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/54371 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/6 shards):   0%|          | 0/54371 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2592 [00:00<?, ? examples/s]

## Merge with generated datasets - now 100% of data

In [7]:
data_init = datasets.load_from_disk("../../data/gov_report/gr_merged_with_gold_with_ids_debug")

In [8]:
gen_data_pathes = [
    "../../data/finetune_generated/gov_rep_base_gen_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_7b_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_32b_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_72b_1",
    "../../data/finetune_generated/gov_rep_base_gen_gemma_2_9b_1",
    "../../data/finetune_generated/gov_rep_base_gen_phi_3_small_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_3_4b_1",
]

In [9]:
# base merging - get 50% of each model, after merge with train, build splits and save

In [10]:
all_gen_data = []
for path in gen_data_pathes:
    gen_data = datasets.load_from_disk(path)
    np.random.seed(43)
    # now select all data
    #gen_data = gen_data.select(np.random.choice(np.arange(len(gen_data)), int(0.5*len(gen_data)), replace=False))
    all_gen_data.append(gen_data)
all_gen_data = datasets.concatenate_datasets(all_gen_data)

In [11]:
for col in data_init["train"].column_names:
    all_gen_data = all_gen_data.cast_column(col, data_init["train"].features[col])

Casting the dataset:   0%|          | 0/68988 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/68988 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/68988 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/68988 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/68988 [00:00<?, ? examples/s]

In [12]:
data_init["train"].features

{'doc_id': Value(dtype='string', id=None),
 'document_sections': {'alignment': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
  'depth': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None),
  'par_lens': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None),
  'paragraphs': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
  'title': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None)},
 'question_summary_pairs': Sequence(feature={'question': Value(dtype='string', id=None), 'summary': Value(dtype='string', id=None), 'parent_pair_index': Value(dtype='int32', id=None), 'summary_paragraph_index': Value(dtype='int32', id=None)}, length=-1, id=None),
 'document': Value(dtype='string', id=None),
 'golden_pars': {'paragraphs': Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None),
  'paragraphs_ids': Sequence(feature=Sequence(feature=Value(dtype='int64

In [13]:
all_gen_data.features

{'doc_id': Value(dtype='string', id=None),
 'document_sections': {'alignment': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
  'depth': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None),
  'par_lens': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None),
  'paragraphs': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
  'title': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None)},
 'question_summary_pairs': Sequence(feature={'question': Value(dtype='string', id=None), 'summary': Value(dtype='string', id=None), 'parent_pair_index': Value(dtype='int32', id=None), 'summary_paragraph_index': Value(dtype='int32', id=None)}, length=-1, id=None),
 'document': Value(dtype='string', id=None),
 'golden_pars': {'paragraphs': Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None),
  'paragraphs_ids': Sequence(feature=Sequence(feature=Value(dtype='int64

In [14]:
data_with_gen = copy.deepcopy(data_init)
data_with_gen["train"] = datasets.concatenate_datasets([data_with_gen["train"], all_gen_data])

In [15]:
data_with_gen["train"] = data_with_gen["train"].shuffle(seed=43)

In [16]:
data_with_gen.save_to_disk("../../data/gov_report/gr_merged_with_gold_with_ids_with_100_gen")

Saving the dataset (0/19 shards):   0%|          | 0/70359 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/171 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/172 [00:00<?, ? examples/s]

## Build splits - from 2k to 16k - for gen data

In [17]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [18]:
def caclulate_par_length(example):
    par_lens = []
    for title, text in zip(example["document_sections"]["title"], example["document_sections"]["paragraphs"]):
        messages = [
            {"role": "system", "content": ''},
            {"role": "user", "content": title + '\n'+ text}
        ]
        length = len(tokenizer.apply_chat_template(messages, tokenize=True))
        par_lens.append(length)
    example["document_sections"]["par_lens"] = par_lens
    return example

In [19]:
dat_with_gold = data_with_gen

In [ ]:
dat_with_gold = dat_with_gold.map(caclulate_par_length)

Map:   0%|          | 0/70359 [00:00<?, ? examples/s]

In [ ]:
def match_gold_par(example):
    gold_pars_ids = []
    for gold_pars in example["golden_pars"]["paragraphs"]:
        buf_gold_ids = []
        for gold_par in gold_pars:
            for par_idx, par in enumerate(example["document_sections"]["paragraphs"]):
                if par == gold_par:
                    buf_gold_ids.append(par_idx)
                    break
        gold_pars_ids.append(buf_gold_ids)
    example["golden_pars"]["paragraphs_ids"] = gold_pars_ids
    return example

In [ ]:
dat_with_gold_ids = dat_with_gold.map(match_gold_par)

In [ ]:
def sample_pars(par_ids, gold_par_ids, par_lens, max_len):
    max_iters = 20
    curr_len = sum(np.array(par_lens)[gold_par_ids])
    sample_length = np.random.randint(min(max_len-1, curr_len), max_len)
    selected_ids = copy.deepcopy(gold_par_ids)
    ids_for_sampling = [x for x in par_ids if x not in gold_par_ids]
    steps = 0
    while curr_len < sample_length and steps < max_iters and len(ids_for_sampling) > 0:
        buf_idx = int(np.random.choice(ids_for_sampling, 1, replace=False)[0])
        buf_len = curr_len + par_lens[buf_idx]
        if buf_len > sample_length:
            steps += 1
        else:
            steps = 0
            curr_len = buf_len
            selected_ids.append(buf_idx)
            ids_for_sampling.remove(buf_idx)
    return selected_ids

In [ ]:
def caclulate_length_full(example):
    messages = [
        {"role": "user", "content": f"Question:\n{example['question']}\nContext:\n{example['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{example['answer']}\n"},
    ]
    length = len(tokenizer.apply_chat_template(messages, tokenize=True))
    example["length"] = length
    return example

In [32]:
x=1

In [ ]:
dataset_2k = {}
max_len = 2048
save_name = "gov_report_0k_2k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 370.29it/s]


Map:   0%|          | 0/88866 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/88866 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/61000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/483 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1258 [00:00<?, ? examples/s]

In [ ]:
dataset_2k = {}
max_len = 4096
save_name = "gov_report_0k_4k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 295.94it/s]


Map:   0%|          | 0/88866 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/88866 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/83489 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/876 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2215 [00:00<?, ? examples/s]

In [ ]:
dataset_2k = {}
max_len = 8192
save_name = "gov_report_0k_8k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 251.22it/s]


Map:   0%|          | 0/88866 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/88866 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/5 shards):   0%|          | 0/88571 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/988 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2543 [00:00<?, ? examples/s]

In [ ]:
dataset_2k = {}
max_len = 4096*4
save_name = "gov_report_0k_16k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 213.34it/s]


Map:   0%|          | 0/88866 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/88866 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/7 shards):   0%|          | 0/88817 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2592 [00:00<?, ? examples/s]

In [ ]:
dataset_2k = {}
max_len = 1024*32
save_name = "gov_report_0k_32k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 201.75it/s]


Map:   0%|          | 0/88866 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/88866 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/9 shards):   0%|          | 0/88854 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2592 [00:00<?, ? examples/s]

In [ ]:
dataset_2k = {}
max_len = 1024*64
save_name = "gov_report_0k_64k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 191.39it/s]


Map:   0%|          | 0/88866 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/88866 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/10 shards):   0%|          | 0/88866 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2592 [00:00<?, ? examples/s]

## Merge with generated datasets - now 100% of data + new models

In [14]:
data_init = datasets.load_from_disk("../../data/gov_report/gr_merged_with_gold_with_ids_debug")

In [15]:
gen_data_pathes = [
    "../../data/finetune_generated/gov_rep_base_gen_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_7b_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_32b_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_72b_1",
    "../../data/finetune_generated/gov_rep_base_gen_gemma_2_9b_1",
    "../../data/finetune_generated/gov_rep_base_gen_phi_3_small_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_3_4b_1",

    "../../data/finetune_generated/gov_rep_base_gen_phi_4_14b_1",
    "../../data/finetune_generated/gov_rep_base_gen_mistral_7b_v3_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_3_30b_a3b_gptq_int4_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_3_8b_1",
]

In [16]:
# base merging - get 50% of each model, after merge with train, build splits and save

In [17]:
all_gen_data = []
for path in gen_data_pathes:
    gen_data = datasets.load_from_disk(path)
    np.random.seed(43)
    # now select all data
    #gen_data = gen_data.select(np.random.choice(np.arange(len(gen_data)), int(0.5*len(gen_data)), replace=False))
    all_gen_data.append(gen_data)
all_gen_data = datasets.concatenate_datasets(all_gen_data)

In [ ]:
for col in data_init["train"].column_names:
    all_gen_data = all_gen_data.cast_column(col, data_init["train"].features[col])

Casting the dataset:   0%|          | 0/108907 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/108907 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/108907 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/108907 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/108907 [00:00<?, ? examples/s]

In [ ]:
data_init["train"].features

{'doc_id': Value(dtype='string', id=None),
 'document_sections': {'alignment': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
  'depth': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None),
  'par_lens': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None),
  'paragraphs': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
  'title': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None)},
 'question_summary_pairs': Sequence(feature={'question': Value(dtype='string', id=None), 'summary': Value(dtype='string', id=None), 'parent_pair_index': Value(dtype='int32', id=None), 'summary_paragraph_index': Value(dtype='int32', id=None)}, length=-1, id=None),
 'document': Value(dtype='string', id=None),
 'golden_pars': {'paragraphs': Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None),
  'paragraphs_ids': Sequence(feature=Sequence(feature=Value(dtype='int64

In [ ]:
all_gen_data.features

{'doc_id': Value(dtype='string', id=None),
 'document_sections': {'alignment': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
  'depth': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None),
  'par_lens': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None),
  'paragraphs': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
  'title': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None)},
 'question_summary_pairs': Sequence(feature={'question': Value(dtype='string', id=None), 'summary': Value(dtype='string', id=None), 'parent_pair_index': Value(dtype='int32', id=None), 'summary_paragraph_index': Value(dtype='int32', id=None)}, length=-1, id=None),
 'document': Value(dtype='string', id=None),
 'golden_pars': {'paragraphs': Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None),
  'paragraphs_ids': Sequence(feature=Sequence(feature=Value(dtype='int64

In [ ]:
data_with_gen = copy.deepcopy(data_init)
data_with_gen["train"] = datasets.concatenate_datasets([data_with_gen["train"], all_gen_data])

In [ ]:
data_with_gen["train"] = data_with_gen["train"].shuffle(seed=43)

In [ ]:
data_with_gen.save_to_disk("../../data/gov_report/gr_merged_with_gold_with_ids_with_100_gen_plus_new_models")

Saving the dataset (0/29 shards):   0%|          | 0/110278 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/171 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/172 [00:00<?, ? examples/s]

In [ ]:
# was 70359/171/172

## Build splits - from 2k to 16k - for gen data

In [25]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [26]:
def caclulate_par_length(example):
    par_lens = []
    for title, text in zip(example["document_sections"]["title"], example["document_sections"]["paragraphs"]):
        messages = [
            {"role": "system", "content": ''},
            {"role": "user", "content": title + '\n'+ text}
        ]
        length = len(tokenizer.apply_chat_template(messages, tokenize=True))
        par_lens.append(length)
    example["document_sections"]["par_lens"] = par_lens
    return example

In [27]:
dat_with_gold = data_with_gen

In [ ]:
dat_with_gold = dat_with_gold.map(caclulate_par_length)

Map:   0%|          | 0/110278 [00:00<?, ? examples/s]

Map:   0%|          | 0/171 [00:00<?, ? examples/s]

Map:   0%|          | 0/172 [00:00<?, ? examples/s]

In [ ]:
def match_gold_par(example):
    gold_pars_ids = []
    for gold_pars in example["golden_pars"]["paragraphs"]:
        buf_gold_ids = []
        for gold_par in gold_pars:
            for par_idx, par in enumerate(example["document_sections"]["paragraphs"]):
                if par == gold_par:
                    buf_gold_ids.append(par_idx)
                    break
        gold_pars_ids.append(buf_gold_ids)
    example["golden_pars"]["paragraphs_ids"] = gold_pars_ids
    return example

In [ ]:
dat_with_gold_ids = dat_with_gold.map(match_gold_par)

Map:   0%|          | 0/110278 [00:00<?, ? examples/s]

Map:   0%|          | 0/171 [00:00<?, ? examples/s]

Map:   0%|          | 0/172 [00:00<?, ? examples/s]

In [ ]:
def sample_pars(par_ids, gold_par_ids, par_lens, max_len):
    max_iters = 20
    curr_len = sum(np.array(par_lens)[gold_par_ids])
    sample_length = np.random.randint(min(max_len-1, curr_len), max_len)
    selected_ids = copy.deepcopy(gold_par_ids)
    ids_for_sampling = [x for x in par_ids if x not in gold_par_ids]
    steps = 0
    while curr_len < sample_length and steps < max_iters and len(ids_for_sampling) > 0:
        buf_idx = int(np.random.choice(ids_for_sampling, 1, replace=False)[0])
        buf_len = curr_len + par_lens[buf_idx]
        if buf_len > sample_length:
            steps += 1
        else:
            steps = 0
            curr_len = buf_len
            selected_ids.append(buf_idx)
            ids_for_sampling.remove(buf_idx)
    return selected_ids

In [ ]:
def caclulate_length_full(example):
    messages = [
        {"role": "user", "content": f"Question:\n{example['question']}\nContext:\n{example['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{example['answer']}\n"},
    ]
    length = len(tokenizer.apply_chat_template(messages, tokenize=True))
    example["length"] = length
    return example

In [ ]:
x=1

In [ ]:
dataset_2k = {}
max_len = 2048
save_name = "gov_report_0k_2k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen_plus/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 348.30it/s]


Map:   0%|          | 0/128785 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/128785 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/90476 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/483 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1259 [00:00<?, ? examples/s]

In [ ]:
dataset_2k = {}
max_len = 4096
save_name = "gov_report_0k_4k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen_plus/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 394.91it/s]


Map:   0%|          | 0/128785 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/128785 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/4 shards):   0%|          | 0/121690 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/876 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2215 [00:00<?, ? examples/s]

In [ ]:
dataset_2k = {}
max_len = 8192
save_name = "gov_report_0k_8k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen_plus/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 329.40it/s]


Map:   0%|          | 0/128785 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/128785 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/7 shards):   0%|          | 0/128432 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/988 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2543 [00:00<?, ? examples/s]

In [ ]:
dataset_2k = {}
max_len = 4096*4
save_name = "gov_report_0k_16k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen_plus/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 273.14it/s]


Map:   0%|          | 0/128785 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/128785 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/10 shards):   0%|          | 0/128734 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2592 [00:00<?, ? examples/s]

In [ ]:
dataset_2k = {}
max_len = 1024*32
save_name = "gov_report_0k_32k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen_plus/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 262.66it/s]


Map:   0%|          | 0/128785 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/128785 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/13 shards):   0%|          | 0/128773 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2592 [00:00<?, ? examples/s]

In [ ]:
dataset_2k = {}
max_len = 1024*64
save_name = "gov_report_0k_64k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen_plus/{save_name}")


100%|██████████| 172/172 [00:00<00:00, 220.64it/s]


Map:   0%|          | 0/128785 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Filter:   0%|          | 0/128785 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

Saving the dataset (0/14 shards):   0%|          | 0/128785 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2592 [00:00<?, ? examples/s]

## Merge with generated datasets - now 100% of data + new models + more data

In [17]:
data_init = datasets.load_from_disk("../../data/gov_report/gr_merged_with_gold_with_ids_debug")

In [21]:
gen_data_pathes = [
    "../../data/finetune_generated/gov_rep_base_gen_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_7b_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_32b_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_72b_1",
    "../../data/finetune_generated/gov_rep_base_gen_gemma_2_9b_1",
    "../../data/finetune_generated/gov_rep_base_gen_phi_3_small_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_3_4b_1",

    "../../data/finetune_generated/gov_rep_base_gen_phi_4_14b_1",
    "../../data/finetune_generated/gov_rep_base_gen_mistral_7b_v3_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_3_30b_a3b_gptq_int4_1",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_3_8b_1",

    
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_7b_2",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_7b_25",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_32b_2",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_72b_2",
    #"../../data/finetune_generated/gov_rep_base_gen_gemma_2_9b_2",
    "../../data/finetune_generated/gov_rep_base_gen_phi_3_small_2",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_3_4b_2",
    "../../data/finetune_generated/gov_rep_base_gen_phi_4_14b_2",
    "../../data/finetune_generated/gov_rep_base_gen_mistral_7b_v3_2",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_3_30b_a3b_gptq_int4_2",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_3_8b_2",


    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_7b_3",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_32b_3",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_25_72b_3",
    "../../data/finetune_generated/gov_rep_base_gen_gemma_2_9b_3",
    "../../data/finetune_generated/gov_rep_base_gen_phi_3_small_3",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_3_4b_3",
    "../../data/finetune_generated/gov_rep_base_gen_phi_4_14b_3",
    "../../data/finetune_generated/gov_rep_base_gen_mistral_7b_v3_3",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_3_30b_a3b_gptq_int4_3",
    "../../data/finetune_generated/gov_rep_base_gen_qwen_3_8b_3",
]

In [22]:
# base merging - get 50% of each model, after merge with train, build splits and save

In [23]:
all_gen_data = []
for path in gen_data_pathes:
    gen_data = datasets.load_from_disk(path)
    np.random.seed(43)
    # now select all data
    #gen_data = gen_data.select(np.random.choice(np.arange(len(gen_data)), int(0.5*len(gen_data)), replace=False))
    all_gen_data.append(gen_data)
all_gen_data = datasets.concatenate_datasets(all_gen_data)

In [24]:
for col in data_init["train"].column_names:
    all_gen_data = all_gen_data.cast_column(col, data_init["train"].features[col])

Casting the dataset:   0%|          | 0/204942 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/204942 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/204942 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/204942 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/204942 [00:00<?, ? examples/s]

In [25]:
data_init["train"].features

{'doc_id': Value(dtype='string', id=None),
 'document_sections': {'alignment': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
  'depth': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None),
  'par_lens': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None),
  'paragraphs': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
  'title': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None)},
 'question_summary_pairs': Sequence(feature={'question': Value(dtype='string', id=None), 'summary': Value(dtype='string', id=None), 'parent_pair_index': Value(dtype='int32', id=None), 'summary_paragraph_index': Value(dtype='int32', id=None)}, length=-1, id=None),
 'document': Value(dtype='string', id=None),
 'golden_pars': {'paragraphs': Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None),
  'paragraphs_ids': Sequence(feature=Sequence(feature=Value(dtype='int64

In [26]:
all_gen_data.features

{'doc_id': Value(dtype='string', id=None),
 'document_sections': {'alignment': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
  'depth': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None),
  'par_lens': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None),
  'paragraphs': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
  'title': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None)},
 'question_summary_pairs': Sequence(feature={'question': Value(dtype='string', id=None), 'summary': Value(dtype='string', id=None), 'parent_pair_index': Value(dtype='int32', id=None), 'summary_paragraph_index': Value(dtype='int32', id=None)}, length=-1, id=None),
 'document': Value(dtype='string', id=None),
 'golden_pars': {'paragraphs': Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None),
  'paragraphs_ids': Sequence(feature=Sequence(feature=Value(dtype='int64

In [27]:
data_with_gen = copy.deepcopy(data_init)
data_with_gen["train"] = datasets.concatenate_datasets([data_with_gen["train"], all_gen_data])

In [28]:
data_with_gen["train"] = data_with_gen["train"].shuffle(seed=43)

In [29]:
data_with_gen.save_to_disk("../../data/gov_report/gr_merged_with_gold_with_ids_with_100_gen_plus_new_models_more_synth")

Saving the dataset (0/54 shards):   0%|          | 0/206313 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/171 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/172 [00:00<?, ? examples/s]

In [ ]:
# was 70359/171/172

## Build splits - from 2k to 16k - for gen data

In [30]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [31]:
def caclulate_par_length(example):
    par_lens = []
    for title, text in zip(example["document_sections"]["title"], example["document_sections"]["paragraphs"]):
        messages = [
            {"role": "system", "content": ''},
            {"role": "user", "content": title + '\n'+ text}
        ]
        length = len(tokenizer.apply_chat_template(messages, tokenize=True))
        par_lens.append(length)
    example["document_sections"]["par_lens"] = par_lens
    return example

In [32]:
dat_with_gold = data_with_gen

In [ ]:
dat_with_gold = dat_with_gold.map(caclulate_par_length)

Map:   0%|          | 0/206313 [00:00<?, ? examples/s]

In [ ]:
def match_gold_par(example):
    gold_pars_ids = []
    for gold_pars in example["golden_pars"]["paragraphs"]:
        buf_gold_ids = []
        for gold_par in gold_pars:
            for par_idx, par in enumerate(example["document_sections"]["paragraphs"]):
                if par == gold_par:
                    buf_gold_ids.append(par_idx)
                    break
        gold_pars_ids.append(buf_gold_ids)
    example["golden_pars"]["paragraphs_ids"] = gold_pars_ids
    return example

In [ ]:
dat_with_gold_ids = dat_with_gold.map(match_gold_par)

In [ ]:
def sample_pars(par_ids, gold_par_ids, par_lens, max_len):
    max_iters = 20
    curr_len = sum(np.array(par_lens)[gold_par_ids])
    sample_length = np.random.randint(min(max_len-1, curr_len), max_len)
    selected_ids = copy.deepcopy(gold_par_ids)
    ids_for_sampling = [x for x in par_ids if x not in gold_par_ids]
    steps = 0
    while curr_len < sample_length and steps < max_iters and len(ids_for_sampling) > 0:
        buf_idx = int(np.random.choice(ids_for_sampling, 1, replace=False)[0])
        buf_len = curr_len + par_lens[buf_idx]
        if buf_len > sample_length:
            steps += 1
        else:
            steps = 0
            curr_len = buf_len
            selected_ids.append(buf_idx)
            ids_for_sampling.remove(buf_idx)
    return selected_ids

In [ ]:
def caclulate_length_full(example):
    messages = [
        {"role": "user", "content": f"Question:\n{example['question']}\nContext:\n{example['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{example['answer']}\n"},
    ]
    length = len(tokenizer.apply_chat_template(messages, tokenize=True))
    example["length"] = length
    return example

In [ ]:
x=1

In [ ]:
dataset_2k = {}
max_len = 2048
save_name = "gov_report_0k_2k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen_plus_plus/{save_name}")


In [ ]:
dataset_2k = {}
max_len = 4096
save_name = "gov_report_0k_4k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen_plus_plus/{save_name}")


In [ ]:
dataset_2k = {}
max_len = 8192
save_name = "gov_report_0k_8k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen_plus_plus/{save_name}")


In [ ]:
dataset_2k = {}
max_len = 4096*4
save_name = "gov_report_0k_16k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen_plus_plus/{save_name}")


In [ ]:
dataset_2k = {}
max_len = 1024*32
save_name = "gov_report_0k_32k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen_plus_plus/{save_name}")


In [ ]:
dataset_2k = {}
max_len = 1024*64
save_name = "gov_report_0k_64k"
np.random.seed(43)
for split in dat_with_gold_ids.keys():
    # build one sample
    data_split = {
        "doc_id": [],
        "context": [],
        "question": [],
        "answer": [],
        "par_ids": [],
        "gold_par_ids": [],
    }
    for el in tqdm(dat_with_gold_ids[split]):
        buf_texts = el["document_sections"]    
        for idx in range(len(el["golden_pars"]["questions"])):
            questions = el["golden_pars"]["questions"][idx]
            answers = el["golden_pars"]["summaries"][idx]
            gold_par_ids = el["golden_pars"]["paragraphs_ids"][idx]
            # build context for each question
            # with sampling 
            text_ids = sample_pars(list(range(len(buf_texts["par_lens"]))), gold_par_ids, buf_texts["par_lens"], max_len)
            text_ids = sorted(text_ids)
            sum_context_len = sum(np.array(buf_texts["par_lens"])[text_ids])
            sum_context = [title + '\n' + text for title, text in zip(np.array(buf_texts["title"])[text_ids], np.array(buf_texts["paragraphs"])[text_ids])]
            sum_context = "\n".join(sum_context).strip("\n")
            # w/o sampling
            #start_idx, end_idx = min(gold_par_ids), max(gold_par_ids)
            #sum_context_len = sum(buf_texts["par_lens"][start_idx:end_idx])
            #sum_context = [title + '\n' + text for title, text in zip(buf_texts["title"][start_idx:end_idx], buf_texts["paragraphs"][start_idx:end_idx])]
            for question, answer in zip(questions, answers):
                data_split["doc_id"].append(el["doc_id"])
                data_split["question"].append(question)
                data_split["answer"].append(answer)
                data_split["context"].append(sum_context)
                data_split["par_ids"].append(text_ids)
                data_split["gold_par_ids"].append(gold_par_ids)
    dataset_2k[split] = datasets.Dataset.from_dict(data_split)

gov_rep_2k = datasets.DatasetDict(dataset_2k)
# subsample val and test to speedup eval
np.random.seed(43)
val_ids = np.random.choice(np.arange(len(gov_rep_2k["validation"])), 1000)
gov_rep_2k["validation"] = gov_rep_2k["validation"].select(val_ids)
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < max_len)
gov_rep_2k.save_to_disk(f"../../data/gov_report_100_gen_plus_plus/{save_name}")


# Analyze dataset

In [34]:
gr_base_2k = datasets.load_from_disk(f"../../data/gov_report/gov_report_0k_2k")

In [36]:
lens_2k = gr_base_2k["train"]["length"]

In [42]:
np.percentile(lens_2k, 90)

np.float64(1935.0)

In [99]:
def caclulate_length_full(example):
    messages = [
        {"role": "user", "content": f"Question:\n{example['question']}\nContext:\n{example['context']}\n"},
        {"role": "assistant", "content": f"Answer:\n{example['answer']}\n"},
    ]
    length = len(tokenizer.apply_chat_template(messages, tokenize=True))
    example["length"] = length
    return example

In [100]:
gov_rep_2k = gov_rep_2k.map(caclulate_length_full)

Map:   0%|          | 0/19878 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

In [101]:
lens = gov_rep_2k["train"]["length"]

In [109]:
np.mean(lens), np.min(lens), np.max(lens), np.percentile(lens, 50)

(np.float64(2587.8853506388973),
 np.int64(187),
 np.int64(50229),
 np.float64(2053.5))

In [112]:
gov_rep_2k = gov_rep_2k.filter(lambda x: x["length"] < 2048)


Filter:   0%|          | 0/19843 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2592 [00:00<?, ? examples/s]

In [113]:
gov_rep_2k

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 9903
    })
    validation: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 483
    })
    test: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 1259
    })
})

In [90]:
len(dataset_2k["validation"])

2585

In [89]:
dataset_2k["validation"][-1]

{'doc_id': 'GAO_GAO-15-191',
 'context': 'Recommendations for Executive Action\nTo help strengthen the ability of agencies to deal with poor performers, we recommend that the Director of OPM, in conjunction with the CHCO Council and, as appropriate, with key stakeholders such as federal employee labor unions, take the following four actions: 1. To more effectively ensure that agencies have a well-qualified cadre of supervisors capable of effectively addressing poor performance, determine if promising practices at some agencies should be more widely used government-wide. Such practices include (1) extending the supervisory probationary period beyond 1-year to include at least one full employee appraisal cycle; (2) providing detail opportunities or rotational assignments to supervisory candidates prior to promotion, where the candidate can develop and demonstrate supervisory competencies; and (3) using a dual career ladder structure as a way to advance employees who may have particular t

In [55]:
dat_with_gold_ids["validation"]["golden_pars"][0]#["paragraphs"]

{'paragraphs': [['During the Clean Air Act debate in 1990 on the Title IV sulfur dioxide (SO 2 ) cap-and trade program, CRS found it difficult to analyze the cost of the bill beyond the first 10 years (1990-2000), and considered any breakdown of even 2000 data on a state-by-state basis as "not useful for any more than illustrative purposes." As stated in 1990:\nIt is difficult (and some would consider it unwise) to project costs up to the year 2000, much less beyond. The already tenuous assumption that current regulatory standards will remain constant becomes more unrealistic, and other unforeseen events (such as electric utility deregulation) loom as critical issues which can not be modeled. Hence, cost projections beyond the year 2000 are at best speculative, and are more a function of each model \' s assumptions and structure than they are of the details of proposed legislation . Projections this far into the future are based more on philosophy than analysis. [emphasis in original]\

In [46]:
dat_with_gold["validation"]["golden_pars"][0]["paragraphs"]

[['During the Clean Air Act debate in 1990 on the Title IV sulfur dioxide (SO 2 ) cap-and trade program, CRS found it difficult to analyze the cost of the bill beyond the first 10 years (1990-2000), and considered any breakdown of even 2000 data on a state-by-state basis as "not useful for any more than illustrative purposes." As stated in 1990:\nIt is difficult (and some would consider it unwise) to project costs up to the year 2000, much less beyond. The already tenuous assumption that current regulatory standards will remain constant becomes more unrealistic, and other unforeseen events (such as electric utility deregulation) loom as critical issues which can not be modeled. Hence, cost projections beyond the year 2000 are at best speculative, and are more a function of each model \' s assumptions and structure than they are of the details of proposed legislation . Projections this far into the future are based more on philosophy than analysis. [emphasis in original]\nThe history of

In [45]:
dat_with_gold["validation"]["document_sections"][1]

{'alignment': ['h0_title h2_title h1_title',
  'h2_full',
  'h0_title h1_title',
  'h1_full',
  'h0_title h1_full',
  '',
  'h0_full',
  'h0_full h1_full',
  'h0_full h1_title',
  '',
  'h1_full',
  '',
  '',
  '',
  'h2_title h1_title',
  'h2_full h1_full',
  ''],
 'depth': [0, 1, 1, 2, 2, 3, 3, 3, 1, 2, 2, 1, 2, 2, 1, 2, 2],
 'par_lens': [31,
  601,
  105,
  345,
  231,
  220,
  562,
  449,
  172,
  517,
  542,
  39,
  260,
  100,
  40,
  640,
  315],
 'paragraphs': ['',
  'The Supplemental Nutrition Assistance Program (SNAP) provided food assistance to a monthly average of 40.3 million persons in 20.1 million households in FY2018. Total benefit costs were $60.6 billion in FY2018.\nSNAP participation and costs increased markedly from FY2007 to FY2013, mostly as a result of automatic and legislated responses to the recession. In FY2014, both participation and costs declined from peak FY2013 levels and have continued to decline. While much of the FY2007 to FY2013 increase in participat

In [ ]:
# now match gold paragraphs with indices
# after randomly select paragraphs by length for each sample in split
# after flatten qa and save split

In [ ]:
for split in dat.keys():
    for el in dat[split]:
        par_samples = par_subset.select(lambda x: x["doc_id"] == el["doc_id"])
        titles = [par_sample["title"] for par_sample in par_samples["document_sections"]]
        pars = [par_sample["paragraphs"] for par_sample in par_samples["document_sections"]]
        quests = [par_sample["question"] for par_sample in par_samples["question_summary_pairs"]]
        summs = [par_sample["summary"] for par_sample in par_samples["question_summary_pairs"]]
        pass

In [ ]:
# Extract paragraph for each QA
# Choose length and concat with other pars
# 

In [8]:
dat["validation"]["document_sections"][0]

{'title': ['',
  'Overview of the Major Provisions of S. 2191 (S. 3036)',
  'Earlier Versions of the Bill',
  'Bill as Introduced',
  'Bill as Reported by Subcommittee',
  'As Ordered Reported by Committee',
  'Deficit Reduction Amendment',
  'Introduction: Models Cannot Predict the Future Costs of a Climate Change Program',
  'Lessons from SO2 Cap and Trade Program',
  'An Illustrative Example from Analyses of S. 2191',
  'Likelihood for More Noise in Greenhouse Gas Reduction Cost Estimates',
  'Complexity of the Problem',
  'Flexibility of Cap-and-Trade Program',
  'Importance of Technology to Future Results',
  'Increasing Problems with Ceteris Paribus Analysis31',
  'Changing Baselines By Changing Laws',
  'Changing Baselines By Changing Regulation35',
  'Measuring the Noise: A Web of Cost Measures',
  'Three Perspectives: Getting Out of the Noise',
  'Results for S. 2191',
  'Impact on Greenhouse Gas Emissions',
  'Impact on Non-Greenhouse Gas Emissions',
  'Impact on GDP Per Capi

In [13]:
dat["validation"]["question_summary_pairs"][0]

{'question': ['What does this report examine?',
  'Why might projections be inaccurate?',
  'How might these studies be helpful?',
  'By what would the ultimate cost of S. 2191 be determined?',
  "Why is it hard to project technology's potential to reduce S. 2191's costs?",
  'What do the cases suggest regarding the use of CCS bonus allowances?',
  "How is S. 2191's climate-related benefit best considered?",
  'What efforts did the US make to engage with the rest of the world on this issue?',
  'What issues does this global context raise?'],
 'summary': ['This report examines six studies that project the costs of S. 2191 (S. 3036) to 2030 or 2050.',
  'It is difficult to project costs up to the year 2030, much less beyond. The already tenuous assumption that regulatory standards will remain constant becomes more unrealistic, and other unforeseen events loom as critical issues which cannot be modeled. Long-term cost projections are at best speculative, and should be viewed with attentiv

## Build gov_rep with prolonged validation, gov-rep-100+

In [61]:
gr_8k = datasets.load_from_disk("../../data/gov_report/gov_report_0k_8k/")
gr_4k = datasets.load_from_disk("../../data/gov_report/gov_report_0k_4k/")
gr_32k = datasets.load_from_disk("../../data/gov_report/gov_report_0k_32k/")

In [62]:
gr_4k["validation"] = gr_32k["validation"]
gr_8k["validation"] = gr_32k["validation"]

In [63]:
gr_8k

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 19682
    })
    validation: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 2543
    })
})

In [64]:
gr_4k

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 17450
    })
    validation: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 2215
    })
})

In [65]:
gr_4k.save_to_disk("../../data/gov_report/gov_report_0k_4k_val_32k/")

Saving the dataset (0/1 shards):   0%|          | 0/17450 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2215 [00:00<?, ? examples/s]

In [60]:
gr_8k.save_to_disk("../../data/gov_report/gov_report_0k_8k_val_32k/")

Saving the dataset (0/1 shards):   0%|          | 0/19682 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2543 [00:00<?, ? examples/s]

In [45]:
lens = gr_32k["validation"]["length"]

In [54]:
np.percentile(lens, 50)

np.float64(8559.0)

In [53]:
np.percentile(gr_8k["validation"]["length"], 90)

np.float64(7173.0)

## Build gov_rep with prolonged validation

In [12]:
gr_8k = datasets.load_from_disk("../../data/gov_report_100_gen_plus/gov_report_0k_8k/")
gr_16k = datasets.load_from_disk("../../data/gov_report_100_gen_plus/gov_report_0k_16k/")

In [13]:
gr_8k["validation"] = gr_16k["validation"]

In [14]:
gr_8k

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 128432
    })
    validation: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['doc_id', 'context', 'question', 'answer', 'par_ids', 'gold_par_ids', 'length'],
        num_rows: 2543
    })
})

In [15]:
gr_8k.save_to_disk("../../data/gov_report_100_gen_plus/gov_report_0k_8k_val_16k/")

Saving the dataset (0/7 shards):   0%|          | 0/128432 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2543 [00:00<?, ? examples/s]

In [16]:
lens = gr_8k["validation"]["length"]

In [22]:
np.percentile(lens, 60)

np.float64(8068.8)

In [53]:
np.percentile(gr_8k["validation"]["length"], 90)

np.float64(7173.0)